# Model Architectures

from transformers import AutoConfig, AutoModel
import pandas as pd

models = [
    "bert-base-uncased",
    "google/bigbird-roberta-base",
    "distilbert-base-uncased",
    "allenai/longformer-base-4096",
    "monologg/bert-base-cased-goemotions-original"
]

results = []

for model_name in models:
    config = AutoConfig.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    
    layers = getattr(config, "num_hidden_layers", "N/A")
    hidden_size = config.hidden_size
    
    intermediate_size = getattr(config, "intermediate_size", hidden_size * 4)
    total_neurons = layers * intermediate_size
    max_seq = config.max_position_embeddings
    params = sum(p.numel() for p in model.parameters()) / 1e6 
    
    results.append({
        "Model": model_name.split('/')[-1],
        "Layers": layers,
        "Hidden Size": hidden_size,
        "Total Neurons": f"{total_neurons:,}",
        "Max Seq": max_seq,
        "Params (M)": f"{params:.1f}M"
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

# Config

In [1]:
import torch, transformers
print("ok", torch.__version__, transformers.__version__)

ok 2.4.1 4.45.0.dev0


In [2]:
import os
import torch
import json
import logging
import time
import tqdm
import numpy as np
import pandas as pd
import pickle
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import neurox.data.extraction.transformers_extractor as transformers_extractor
from neurox.data.writer import ActivationsWriter
import neurox.data.loader as data_loader
from transformers import AutoConfig
from tqdm import tqdm
import neurox.interpretation.linear_probe as linear_probe
import neurox.interpretation.utils as utils
import neurox.analysis.visualization as TransformersVisualizer
from sklearn.model_selection import train_test_split
from IPython.display import display
import neurox.interpretation.probeless as probeless
from neurox.interpretation.probeless import (
    get_neuron_ordering,
    get_neuron_ordering_for_all_tags
)
import ast
from torch.cuda.amp import autocast
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from matplotlib_venn import venn2
from neurox.interpretation.linear_probe import get_top_neurons
from sklearn.utils import shuffle

from sklearn.model_selection import StratifiedShuffleSplit
from transformers import AutoConfig
from lumia_synapse_api import (
    lumia_layerwise_auc_from_activations,
    lumia_neuron_ranking_and_mask_from_layer
)



In [3]:
logger = logging.getLogger("synapse_logger")
logger.setLevel(logging.INFO)

if not logger.hasHandlers():

    
    file_handler = logging.FileHandler("logs/synapse_extraction_csv_pth.log", mode="w")
    file_handler.setLevel(logging.INFO)

    
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)

    
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)

    
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

logger.info("🚀 Logging configured")


2026-02-20 17:11:18,275 - INFO - 🚀 Logging configured


## Model configuration

In [4]:
# Select the model (options: "BERT", "BigBird", "DistilBERT", "Longformer")
MODEL = "BigBird"

# Paths based on model name
BASE_PATH = f"data/{MODEL}"
input_csv = f"{BASE_PATH}/{MODEL}_tokens_PT.csv"
output_csv = f"{BASE_PATH}/reduced/{MODEL}_tokens_reduced.csv"
labels_output_path = f"{BASE_PATH}/labels_numeric.txt"
label_mapping_path = f"{BASE_PATH}/labels_mapping.json"
activations_file = f"{BASE_PATH}/activations.json"
weights_path = f"{BASE_PATH}/best_model_{MODEL}.pth"

NUM_LABELS = 5 # Normal, Bvdl, RansomwarePoC, TheTick, 



MODEL_HF = {
    "BERT": "bert-base-uncased",
    "BigBird": "google/bigbird-roberta-base",
    "DistilBERT": "distilbert-base-uncased",
    "Longformer": "allenai/longformer-base-4096"
}[MODEL]


device = torch.device("cpu")

## Load Model

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_HF, num_labels=NUM_LABELS)


Some weights of BigBirdForSequenceClassification were not initialized from the model checkpoint at google/bigbird-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:

state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print(f"✅ Loaded {MODEL} with pretrained weights on {device}")

/var/folders/y4/zbxddgn52lx1dx5ktc7jch6m0000gn/T/ipykernel_52065/1331205247.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weights_path, map_loc

✅ Loaded BigBird with pretrained weights on cpu


# Dataset

## Load Dataset

In [7]:
reduction_ratio = 0.001

# skip dataset reduction if already available
if os.path.exists(output_csv) and os.path.exists(labels_output_path):
    logger.info(f"⚡ Reduced dataset found: {output_csv}. Skipping reduction.")
    df_reduced = pd.read_csv(output_csv)
    with open(labels_output_path, "r") as f:
        labels = [int(line.strip()) for line in f]  # Labels as integers
    with open(label_mapping_path, "r") as f:
        label_mapping = json.load(f)  # Load label mapping
else:
    logger.info(f"🔄 Loading dataset from {input_csv}")

    chunk_size = 5000 
    total_rows = sum(1 for _ in open(input_csv)) - 1  # Total rows excluding header
    df_chunks = []

    logger.info(f"🔄 Processing {total_rows} rows in chunks of {chunk_size}...")

    with tqdm(total=total_rows, desc="Processing rows", unit=" rows") as pbar:
        for chunk in pd.read_csv(input_csv, chunksize=chunk_size):
            # Convert `input_ids` from string to list of integers
            chunk['input_ids'] = chunk['input_ids'].apply(lambda x: list(map(int, x.strip("[]").split(","))))
            df_chunks.append(chunk)
            pbar.update(len(chunk))

    df = pd.concat(df_chunks, ignore_index=True)

    # encode labels as integers
    df['label'], unique_labels = pd.factorize(df["label"])
    label_mapping = {label: int(idx) for idx, label in enumerate(unique_labels)}

    
    # Reduce dataset maintaining class proportions
    df_reduced, _ = train_test_split(df, train_size=reduction_ratio, stratify=df["label"], random_state=42)
    labels = df_reduced["label"].tolist()

    # Save reduced dataset and labels
    df_reduced.to_csv(output_csv, index=False)
    with open(labels_output_path, "w") as f:
        for label in labels:
            f.write(str(label) + "\n")

    with open(label_mapping_path, "w") as f:
        json.dump(label_mapping, f, indent=4)

    logger.info(f"✅ Reduced dataset saved to {output_csv}")
    logger.info(f"✅ Numeric labels saved to {labels_output_path}")
    logger.info(f"✅ Label mapping saved to {label_mapping_path}")


2026-02-20 17:11:19,587 - INFO - ⚡ Reduced dataset found: data/BigBird/reduced/BigBird_tokens_reduced.csv. Skipping reduction.


In [8]:

# Create DataLoader

class SyscallDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.data.iloc[idx]['input_ids'])
        label = torch.tensor(self.data.iloc[idx]['label'])
        return input_ids, label

# Initialize DataLoader with reduced dataset
dataset = SyscallDataset(df_reduced)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

logger.info("✅ Dataloader created")

# Ensure `input_ids` are lists of integers
if isinstance(df_reduced["input_ids"].iloc[0], str):
    df_reduced["input_ids"] = df_reduced["input_ids"].apply(lambda x: list(map(int, x.strip("[]").split(","))))


2026-02-20 17:11:19,607 - INFO - ✅ Dataloader created


# Explainability Module

## Extract or load Activations

In [9]:
if os.path.exists(activations_file):
    logger.info(f"⚡ Activations file found: {activations_file}. Skipping extraction.")
else:
    transformers_extractor.extract_representations(
        model, 
        df_reduced["input_ids"].tolist(),  # Pass preprocessed tokens directly
        activations_file,
        device=device,
    )

    logger.info(f"✅ Activations saved to {activations_file}")


activations, num_layers = data_loader.load_activations(activations_file)
logger.info(f"✅ Loaded activations from {activations_file} with {num_layers} layers")

# Load sentence-level classification data using activations
tokens = data_loader.load_sentence_data(
    output_csv, labels_output_path, activations
)

# Create sentence-level tensors for classification
X, y, mapping = utils.create_tensors(
    tokens,
    activations,
    task_specific_tag="NN",
    task_type="classification"
)

label2idx, idx2label, src2idx, idx2src = mapping
logger.info("✅ Created input/output tensors and label mappings for classification")


2026-02-20 17:11:19,628 - INFO - ⚡ Activations file found: data/BigBird/activations.json. Skipping extraction.
2026-02-20 17:11:19,679 - INFO - ✅ Loaded activations from data/BigBird/activations.json with 12 layers
2026-02-20 17:11:19,689 - INFO - ✅ Created input/output tensors and label mappings for classification


Loading json activations from data/BigBird/activations.json...
50 12.0
Number of tokens:  50
length of source dictionary:  17
length of target dictionary:  5
50
Total instances: 50
['s']
Number of samples:  50
Stats: Labels with their frequencies in the final set
2 9
4 9
0 7
1 10
3 15


## Train linear probe

In [10]:
probe = linear_probe.train_logistic_regression_probe(X, y, lambda_l1=0.001, lambda_l2=0.001)
scores = linear_probe.evaluate_probe(probe, X, y, idx_to_class=idx2label)
logger.info(f"🎯 Probe evaluation results: {scores}")

top_neurons_probe, per_class_top_neurons = linear_probe.get_top_neurons(probe, percentage=0.1, class_to_idx=label2idx)
logger.info(f"🔍 Top global neurons: {top_neurons_probe}")
logger.info(f"🔍 Top neurons per class: {per_class_top_neurons}")

Clases en y_train: [0 1 2 3 4]
Training classification probe
Creating model...
Number of training instances: 50
Number of classes: 5


epoch [1/10]: 0it [00:00, ?it/s]

Epoch: [1/10], Loss: 0.0767


epoch [2/10]: 0it [00:00, ?it/s]

Epoch: [2/10], Loss: 0.0437


...

epoch [10/10]: 0it [00:00, ?it/s]

Epoch: [10/10], Loss: 0.0142


Evaluating: 0it [00:00, ?it/s]

2026-02-20 17:11:19,993 - INFO - 🎯 Probe evaluation results: {'__OVERALL__': 0.96, '2': 0.8888888888888888, '4': 1.0, '0': 1.0, '1': 1.0, '3': 0.9333333333333333}
2026-02-20 17:11:19,996 - INFO - 🔍 Top global neurons: [6152 8203 8204 ... 8182 8189 8190]
2026-02-20 17:11:19,998 - INFO - 🔍 Top neurons per class: {'2': array([3846, 5766, 9040, 9096, 7830, 8973, 6513, 8932, 7795, 9124, 8710,
       3855, 9052, 8857, 8607, 8892, 8991, 5000, 6303, 9206, 8966, 6511,
       8613, 9028, 6700, 5486, 7528, 8586, 8889, 8790, 8955, 8527, 6629,
       8965, 8918, 8628, 8499, 2970, 7958, 8485, 9203, 6336, 7896, 6280,
       7891, 8677, 8480, 8827, 8329, 8761, 8567, 7796, 8616, 9186, 4760,
       8694, 6285, 9157, 8951, 5614, 6560, 8569, 4415, 8078, 9022, 8019,
       7693, 9030, 8466, 6022, 8834, 4280, 8847, 8843, 8611, 8048, 4642,
       8909, 4586, 7727, 3104, 7051, 8622, 9151, 7783, 8254, 6790, 3520,
       8326, 9023, 9049, 7449, 7770, 8838, 9196, 8546, 7714, 5030, 7895,
       8773, 8617, 5117, 

Score (accuracy) of the probe: 0.96


# Baseline + Reload

In [11]:
logger = logging.getLogger(__name__)

df = pd.read_csv(input_csv)

# 🎯 Select 50 random examples and reset index
sample_df = df.sample(n=50, random_state=42).reset_index(drop=True)

# 🧹 Convert "input_ids" and "attention_mask" from string to list format
def parse_list(x):
    return ast.literal_eval(x)

sample_df['input_ids'] = sample_df['input_ids'].apply(parse_list)
sample_df['attention_mask'] = sample_df['attention_mask'].apply(parse_list)

# Encode labels to integers
label_encoder = LabelEncoder()
sample_df['label'] = label_encoder.fit_transform(sample_df['label'])
labels_list = sample_df['label'].tolist()

predictions_list = []
model.eval()
torch.cuda.empty_cache()

def quick_baseline_f1(model, sample_df, labels_list):
    
    model.eval()
    preds = []
    for row in sample_df.itertuples():
        input_ids = torch.tensor(row.input_ids).unsqueeze(0).to(model.device)
        att_mask  = torch.tensor(row.attention_mask).unsqueeze(0).to(model.device)
        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=att_mask).logits
        preds.append(int(logits.argmax(dim=-1)))
    f1w = f1_score(labels_list, preds, average="weighted", zero_division=0)
    logger.info(f"[Baseline Check] Weighted F1-score: {f1w:.4f}")
    return f1w

In [12]:
quick_baseline_f1(model, sample_df, labels_list)

0.8306384351683807

# Intervention Module

## Function definition

In [13]:
def get_top_k_neurons_exact(probe, percentage: float) -> list[int]:
    """
    Return exactly N = round(total_neurons * percentage) neuron indices, sorted by importance.
    Importance is measured as the sum of absolute values of weights across all output classes.
    """
    weight_matrix = probe.linear.weight.detach().abs()  # [num_classes, num_neurons]
    importance = weight_matrix.sum(dim=0).cpu().numpy()  # [num_neurons]
    total_neurons = len(importance)
    top_n = round(total_neurons * percentage)

    sorted_indices = importance.argsort()[-top_n:]  # Top-N by importance
    return sorted_indices.tolist()

In [14]:
def make_cls_silence_hook(indices):
    """
    Forward hook that zeros out the selected neuron indices in the CLS token.
    It is compatible with:
      - BERT / RoBERTa / BigBird / Longformer (tensor output)
      - DistilBERT (tuple output, usually (hidden_state,) or (hidden_state, attentions))
    """
    idxs = torch.tensor(indices, dtype=torch.long)

    def hook(module, inp, output):
        # 1) Unify output into a `hidden` tensor
        if isinstance(output, tuple):
            if len(output) == 0:
                # Nothing to do
                return output
            hidden = output[0]
        else:
            hidden = output

        # 2) Sanity checks
        if not hasattr(hidden, "dim") or hidden.dim() != 3:
            # Not a (batch, seq_len, hidden) tensor → do nothing
            return output

        if idxs.numel() == 0:
            # No neuron to silence in this layer
            return output

        # 3) Clone and modify CLS token
        new_hidden = hidden.clone()
        cls_token = new_hidden[:, 0, :]           # (batch, hidden_dim)
        cls_token[:, idxs] = 0.0                  # silence selected neurons
        new_hidden[:, 0, :] = cls_token

        # 4) Rebuild structure depending on model
        if isinstance(output, tuple):
            # Keep any extra elements (e.g. attentions) untouched
            return (new_hidden,) + tuple(output[1:])
        else:
            return new_hidden

    return hook

In [15]:

def get_encoder_layers(model):
    if hasattr(model, "bert"):
        return model.bert.encoder.layer
    elif hasattr(model, "longformer"):
        return model.longformer.encoder.layer
    elif hasattr(model, "distilbert"):
        return model.distilbert.transformer.layer
    else:
        raise NotImplementedError("❌ Unsupported model architecture.")

# cosas

In [16]:
# TEMP membership labels (hasta que llegue train/test real)
y_task = np.array(y)
N = len(y_task)

p_member = 0.5
seed_mem = 42

sss = StratifiedShuffleSplit(n_splits=1, train_size=p_member, random_state=seed_mem)
member_idx, _ = next(sss.split(np.zeros_like(y_task), y_task))

y_mem = np.zeros(N, dtype=np.int64)
y_mem[member_idx] = 1

logger.info(f"✅ TEMP y_mem built. members={y_mem.sum()} nonmembers={(y_mem==0).sum()}")

In [17]:
import os

L = int(num_layers)
H = int(model.config.hidden_size)
N = len(activations)

A = np.zeros((N, L, H), dtype=np.float32)
for i in range(N):
    flat = activations[i][0]       # [L*H]
    A[i] = flat.reshape(L, H)

logger.info(f"✅ Built A with shape {A.shape} (N,L,H)")

out_dir = f"{BASE_PATH}/results"
os.makedirs(out_dir, exist_ok=True)

In [18]:
df_auc, l_star = lumia_layerwise_auc_from_activations(
    activations=A,
    membership_labels=y_mem,
    seed=123,
    test_size=0.2,
    out_csv=f"{out_dir}/auc_by_layer.csv"
)

logger.info(f"✅ Phase1 done. l_star={l_star}")
display(df_auc.head(10))

,layer_id,auc
9,9,0.52
0,0,0.44
2,2,0.40
1,1,0.36
3,3,0.36
10,10,0.36
8,8,0.28
4,4,0.28
6,6,0.24
7,7,0.24


In [19]:
mask_obj = lumia_neuron_ranking_and_mask_from_layer(
    activations=A,
    membership_labels=y_mem,
    layer_id=l_star,
    top_p=0.01,
    seed=123,
    test_size=0.2,
    out_scores_csv=f"{out_dir}/neuron_scores_layer{l_star}.csv",
    out_mask_json=f"{out_dir}/mask_M_layer{l_star}_top0p01.json"
)

logger.info(f"✅ Phase2 done. mask size={mask_obj['top_k']}")